# Phase 3: Credit Risk Model Development
## Step 1: Baseline Logistic Regression Model
### Credit Risk & Loan Decision Intelligence Platform

---

## SECTION 1 — Phase 3 Introduction

### 1. What is Credit Risk Modeling?
In retail banking, digital micro-finance, and fintech lending, **Credit Risk Modeling** is the practice of using statistical and machine learning methods to determine the probability that a prospective borrower will fail to meet contractual loan repayment obligations. Missing scheduled payments beyond regulatory thresholds (e.g., 90 days past due) is categorized as **default**, causing direct capital loss and increasing Non-Performing Assets (NPAs).

### 2. What is this Model Predicting?
The model is a binary classifier predicting the target variable **`default`**:
- **`0` = Non-default**: The applicant is creditworthy and repays scheduled obligations on time.
- **`1` = Default**: The applicant is high-risk and fails to service their debt obligation.

In addition to discrete 0/1 classification, the model computes an **estimated Probability of Default (PD)** $P(y=1|X) \in [0.0, 1.0]$.

### 3. What is a Baseline Model?
A **baseline model** is a simple, interpretable, and reproducible benchmark model that serves as the foundation for iterative machine learning development. In professional data science, we never deploy complex ensemble or deep learning architectures without first building a robust baseline. Every subsequent candidate model (Random Forest, XGBoost, LightGBM) must demonstrate measurable, statistically sound performance gains over this baseline to justify its operational complexity.

### 4. Why Use Logistic Regression as the First Baseline?
1. **Direct Probability Output**: Logistic Regression applies the standard sigmoid link function $\sigma(z) = \frac{1}{1 + e^{-z}}$ to directly output estimated probabilities of default $\hat{P}(y=1|X)$. *(Note: Standard `predict_proba()` outputs estimated probabilities; formal calibration methods are explored in later stages if required.)*
2. **Interpretability & Explainability**: Learned linear coefficients provide transparent directional feature weights, essential for regulatory compliance (e.g., RBI guidelines, adverse action notices).
3. **Computational Efficiency**: Convex optimization guarantees convergence without expensive hyperparameter searches.
4. **Standard Industry Benchmark**: Logistic Regression is the universally accepted baseline standard across global credit risk and Basel scorecard frameworks.

---
## SECTION 2 — Import Libraries

We import only the core scientific and machine learning libraries required for Phase 3 Step 1 baseline development.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Configure pandas table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Numpy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")


Numpy version  : 2.4.6
Pandas version : 3.0.3


---
## SECTION 3 — Load / Reuse Phase 2 Data

Because Jupyter notebook kernels run independently, we load `credit_risk_50.csv` and reproduce the exact, certified Phase 2 transformation pipeline:
- **Stratified 80/20 train/test split** with `random_state=42`.
- **`StandardScaler`** for the 13 numerical features (fitted strictly on `X_train`).
- **`OneHotEncoder(handle_unknown='ignore', sparse_output=False)`** for `employment_type`.
- Strict zero-leakage enforcement: `X_test` transformed only with training-fitted parameters.
- Matrix dimensional verification: Exactly **14 raw features $\rightarrow$ 16 processed features**.

In [2]:
# Load canonical dataset from data/raw
data_path = '../data/raw/credit_risk_50.csv'
df = pd.read_csv(data_path)

# Separate target y and features X
target_col = 'default'
id_col = 'applicant_id'

y = df[target_col].copy()
ml_feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
X = df[ml_feature_cols].copy()

# Stratified Train/Test Split (80% Train [40 rows], 20% Test [10 rows])
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Define feature groupings exactly as certified in Phase 2
numerical_cols = [
    'age', 'monthly_income', 'loan_amount', 'loan_term_months', 'existing_emi',
    'credit_score', 'credit_utilization_pct', 'late_payment_count', 'previous_defaults',
    'employment_years', 'average_bank_balance', 'monthly_expenses', 'debt_to_income_ratio'
]
categorical_cols = ['employment_type']

# Construct ColumnTransformer matching Phase 2 specifications
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Fit preprocessor strictly on X_train (Zero Data Leakage Rule)
preprocessor.fit(X_train)
transformed_feature_names = list(preprocessor.get_feature_names_out())

# Transform both splits using parameters learned strictly on X_train
X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=transformed_feature_names,
    index=X_train.index
)
X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=transformed_feature_names,
    index=X_test.index
)

print(f"X_train_processed shape : {X_train_processed.shape} (Expected: 40 rows, 16 features)")
print(f"X_test_processed shape  : {X_test_processed.shape}  (Expected: 10 rows, 16 features)")
print(f"y_train shape           : {y_train.shape} (Expected: 40 labels)")
print(f"y_test shape            : {y_test.shape}  (Expected: 10 labels)\n")

# Verify dimensional integrity against Phase 2 requirements
assert X_train_processed.shape == (40, 16), "X_train_processed shape mismatch"
assert X_test_processed.shape == (10, 16), "X_test_processed shape mismatch"
assert y_train.shape == (40,), "y_train shape mismatch"
assert y_test.shape == (10,), "y_test shape mismatch"

print("Feature Names (16 processed features):")
for i, fname in enumerate(transformed_feature_names, 1):
    print(f"  {i:2d}. {fname}")

print("\nPhase 2 Data Loading & Pipeline Reproduction: SUCCESS")


X_train_processed shape : (40, 16) (Expected: 40 rows, 16 features)
X_test_processed shape  : (10, 16)  (Expected: 10 rows, 16 features)
y_train shape           : (40,) (Expected: 40 labels)
y_test shape            : (10,)  (Expected: 10 labels)

Feature Names (16 processed features):
   1. age
   2. monthly_income
   3. loan_amount
   4. loan_term_months
   5. existing_emi
   6. credit_score
   7. credit_utilization_pct
   8. late_payment_count
   9. previous_defaults
  10. employment_years
  11. average_bank_balance
  12. monthly_expenses
  13. debt_to_income_ratio
  14. employment_type_Business
  15. employment_type_Salaried
  16. employment_type_Self-employed

Phase 2 Data Loading & Pipeline Reproduction: SUCCESS


---
## SECTION 4 — Model Training

### Configuration:
We instantiate `sklearn.linear_model.LogisticRegression` with:
- `max_iter=1000`: Ensures complete convergence of the optimization solver (L-BFGS).
- `random_state=42`: Enforces exact reproducibility.

### Strict Test Isolation:
Test isolation is enforced structurally through code architecture: the model is fitted **strictly on `(X_train_processed, y_train)`**. `X_test_processed` and `y_test` are never referenced or touched during the fitting call and are only loaded for downstream evaluation.

In [3]:
# Instantiate baseline Logistic Regression model
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Fit strictly on training data (Test data is never exposed during fitting)
baseline_model.fit(X_train_processed, y_train)

print("Baseline Logistic Regression Model successfully fitted on X_train_processed.")
print(f"Fitted classes                  : {baseline_model.classes_}")
print(f"Number of learned coefficients  : {len(baseline_model.coef_[0])}")
print(f"Model Intercept (bias term)     : {baseline_model.intercept_[0]:.4f}")


Baseline Logistic Regression Model successfully fitted on X_train_processed.
Fitted classes                  : [0 1]
Number of learned coefficients  : 16
Model Intercept (bias term)     : -1.4273


---
## SECTION 5 — Predictions

We generate two distinct prediction outputs on the unseen test set (`X_test_processed`):
1. **`y_test_pred` (Hard Class Predictions)**: Discrete binary outputs $\in \{0, 1\}$ applying the standard 0.50 probability cutoff.
2. **`y_test_proba` (Estimated Probabilities)**: Continuous probability $\hat{P}(y=1|X) \in [0.0, 1.0]$ extracted from column index 1 of `predict_proba()`.

> **Meaning of `y_test_proba`:**  
> `y_test_proba` represents the model's **estimated Probability of Default (PD)** for each applicant. Unlike a binary classification label, PD provides a continuous measure of estimated repayment risk.

In [4]:
# Generate discrete binary class predictions (0 or 1)
y_test_pred = baseline_model.predict(X_test_processed)

# Generate continuous estimated probability of default (column 1)
y_test_proba = baseline_model.predict_proba(X_test_processed)[:, 1]

print("Test Set Prediction Outputs:")
print(f"y_test_pred  (Predicted Class)           : {y_test_pred}")
print(f"y_test_proba (Estimated Default Prob PD): {np.round(y_test_proba, 4)}")


Test Set Prediction Outputs:
y_test_pred  (Predicted Class)           : [0 0 1 0 0 1 1 1 0 0]
y_test_proba (Estimated Default Prob PD): [9.330e-02 2.200e-03 9.746e-01 6.370e-02 5.000e-04 9.704e-01 9.791e-01
 9.210e-01 6.000e-04 1.755e-01]


---
## SECTION 6 — Baseline Evaluation

We calculate actual classification performance metrics on the test partition without hardcoded assumptions:
- **Accuracy**: Overall fraction of correct predictions: $\frac{TP + TN}{TP + TN + FP + FN}$.
- **Precision**: Fraction of predicted defaults that were actual defaults: $\frac{TP}{TP + FP}$.
- **Recall**: Fraction of actual defaults correctly identified: $\frac{TP}{TP + FN}$.
- **F1-Score**: Harmonic mean of precision and recall: $\frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$.
- **ROC-AUC**: Area Under the ROC Curve, computed using continuous estimated probabilities (`y_test_proba`), measuring ranking capability.

> **⚠️ Note on Small-Sample Results:**  
> Any metric achieving 1.0000 on this 10-record educational test set is an observed result of the small, well-separated synthetic sample. It must **NOT** be interpreted as production-level model performance. Real-world credit portfolios contain noise, boundary cases, and unobserved shocks.

In [5]:
# Compute actual evaluation metrics dynamically from predictions
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_proba)

# Construct dynamic metrics summary table
metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Value (Raw)': [accuracy, precision, recall, f1, roc_auc],
    'Value (%)': [f"{accuracy*100:.2f}%", f"{precision*100:.2f}%", f"{recall*100:.2f}%", f"{f1*100:.2f}%", f"{roc_auc*100:.2f}%"]
})

print("=== Baseline Logistic Regression Performance Metrics (Actual Observed) ===")
display(metrics_table)

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual Non-Default (0)', 'Actual Default (1)'],
    columns=['Predicted Non-Default (0)', 'Predicted Default (1)']
)
print("\n=== Confusion Matrix ===")
display(cm_df)

# Detailed classification report
print("\n=== Classification Report ===")
print(classification_report(y_test, y_test_pred, target_names=['Non-Default (0)', 'Default (1)']))


=== Baseline Logistic Regression Performance Metrics (Actual Observed) ===


,Metric,Value (Raw),Value (%)
0,Accuracy,1.0,100.00%
1,Precision,1.0,100.00%
2,Recall,1.0,100.00%
3,F1-Score,1.0,100.00%
4,ROC-AUC,1.0,100.00%



=== Confusion Matrix ===


,Predicted Non-Default (0),Predicted Default (1)
Actual Non-Default (0),6,0
Actual Default (1),0,4



=== Classification Report ===
                 precision    recall  f1-score   support

Non-Default (0)       1.00      1.00      1.00         6
    Default (1)       1.00      1.00      1.00         4

       accuracy                           1.00        10
      macro avg       1.00      1.00      1.00        10
   weighted avg       1.00      1.00      1.00        10



---
## SECTION 7 — Probability of Default Table

We construct a structured evaluation table comparing actual test labels, model binary predictions, and continuous estimated Probability of Default (PD) values across all 10 test applicants.

*(Note: `applicant_id` is excluded from model features and is not exposed as an input.)*

In [6]:
# Construct Estimated Probability of Default comparison DataFrame
pd_summary_df = pd.DataFrame({
    'Test Record Index': X_test_processed.index,
    'Actual Default': y_test.values,
    'Predicted Default': y_test_pred,
    'Estimated PD': np.round(y_test_proba, 4),
    'Risk Category': ['High Risk (Default)' if p >= 0.5 else 'Low/Moderate Risk (Repay)' for p in y_test_proba]
})

print("=== Estimated Probability of Default Table (Test Set: 10 Records) ===")
display(pd_summary_df)


=== Estimated Probability of Default Table (Test Set: 10 Records) ===


,Test Record Index,Actual Default,Predicted Default,Estimated PD,Risk Category
0,10,0,0,0.0933,Low/Moderate Risk (Repay)
1,23,0,0,0.0022,Low/Moderate Risk (Repay)
2,38,1,1,0.9746,High Risk (Default)
3,45,0,0,0.0637,Low/Moderate Risk (Repay)
4,12,0,0,0.0005,Low/Moderate Risk (Repay)
5,40,1,1,0.9704,High Risk (Default)
6,24,1,1,0.9791,High Risk (Default)
7,35,1,1,0.9210,High Risk (Default)
8,16,0,0,0.0006,Low/Moderate Risk (Repay)
9,31,0,0,0.1755,Low/Moderate Risk (Repay)


---
## SECTION 8 — Basic Sanity Checks & Automated Assertions

We execute automated assertions to certify the mathematical integrity, stability, and proper structural execution of the baseline model.

In [7]:
print("Executing Phase 3 Step 1 Automated Sanity Checks...")

# 1. Model is fitted
assert hasattr(baseline_model, 'classes_'), "Check 1 Failed: Model is not fitted!"

# 2. Number of model coefficients = 16
assert len(baseline_model.coef_[0]) == 16, f"Check 2 Failed: Expected 16 coefficients, got {len(baseline_model.coef_[0])}"

# 3. y_test_pred has 10 predictions
assert len(y_test_pred) == 10, f"Check 3 Failed: Expected 10 predictions, got {len(y_test_pred)}"

# 4. y_test_proba has 10 probabilities
assert len(y_test_proba) == 10, f"Check 4 Failed: Expected 10 probabilities, got {len(y_test_proba)}"

# 5. Every probability is between 0 and 1
assert np.all((y_test_proba >= 0.0) & (y_test_proba <= 1.0)), "Check 5 Failed: Probabilities outside [0, 1] bounds!"

# 6. No NaN probabilities
assert not np.isnan(y_test_proba).any(), "Check 6 Failed: NaN detected in predicted probabilities!"

# 7. No infinite probabilities
assert not np.isinf(y_test_proba).any(), "Check 7 Failed: Infinite values detected in predicted probabilities!"

# 8. ROC-AUC calculation succeeds and is a valid float
assert isinstance(roc_auc, (float, np.floating)) and not np.isnan(roc_auc), "Check 8 Failed: ROC-AUC calculation failed!"

# 9. Model fitted feature count matches 16
assert baseline_model.n_features_in_ == 16, "Check 9 Failed: Feature count during fitting != 16!"

# 10. Model input feature count in train and test datasets equals 16
assert X_train_processed.shape[1] == 16 and X_test_processed.shape[1] == 16, "Check 10 Failed: Processed matrices feature count != 16!"

print("=" * 60)
print("PHASE 3 — STEP 1 BASELINE MODEL: VALIDATION PASSED")
print("=" * 60)


Executing Phase 3 Step 1 Automated Sanity Checks...
PHASE 3 — STEP 1 BASELINE MODEL: VALIDATION PASSED


---
## SECTION 9 — Baseline Interpretation

### 1. What Did Logistic Regression Learn?
Logistic Regression learns linear weights (coefficients $\beta_i$) that map the 16 normalized features to the log-odds of defaulting on the loan:

$$\ln\left(\frac{P(\text{Default})}{1 - P(\text{Default})}\right) = \beta_0 + \sum_{i=1}^{16} \beta_i X_i$$

- **Positive coefficients ($\beta_i > 0$)**: Indicate risk escalators that increase default probability (e.g., `debt_to_income_ratio`, `late_payment_count`, `previous_defaults`, `credit_utilization_pct`).
- **Negative coefficients ($\beta_i < 0$)**: Indicate solvency anchors that decrease default probability (e.g., `credit_score`, `monthly_income`, `average_bank_balance`, `employment_years`).

### 2. What Does Estimated Probability of Default (PD) Mean?
PD is the estimated probability that a borrower will fail to repay their loan obligations. It provides a granular, continuous risk measure ($0.0 \le PD \le 1.0$) rather than a simplistic binary classification.

### 3. Difference Between Predicted Class vs. Predicted Probability
- **Predicted Class (`0` or `1`)**: A discrete label generated by applying an arbitrary decision boundary (e.g., $PD \ge 0.50$). It collapses all risk variations into two rigid categories.
- **Predicted Probability (`0.00` to `1.00`)**: The continuous estimated risk score. For example, an applicant with estimated $PD = 0.0022$ and an applicant with estimated $PD = 0.1755$ are both classified as `0` under a 0.50 threshold, but the second applicant carries **~80 times higher default risk**.

### 4. Why Probability is More Useful than Binary Classification in Credit Underwriting
Modern retail credit systems use continuous PD scores to execute sophisticated, automated underwriting rules:
1. **Risk-Based Pricing**: Charging interest rate spreads proportional to default risk (low PD $\rightarrow$ low interest rates; moderate PD $\rightarrow$ risk premium).
2. **Dynamic Credit Limits**: Allocating higher loan amounts to very low PD applicants and restricted credit lines to borderline applicants.
3. **Multi-Tier Routing**: Auto-approving $PD < 0.10$, routing $0.10 \le PD \le 0.30$ to senior underwriting officers for manual review, and auto-declining $PD > 0.30$.
4. **Regulatory Capital Calculation**: Calculating Expected Loss ($\text{EL} = \text{PD} \times \text{EAD} \times \text{LGD}$) under Basel regulatory frameworks.

### 5. Why is This Only a Baseline (Not Final Production Model)?
- **Linearity Restriction**: Logistic Regression assumes linear decision boundaries in log-odds space and cannot capture complex non-linear interactions without explicit manual engineering.
- **Small Educational Sample**: Although the model scored perfectly on our 10-record synthetic test split, real-world production portfolios exhibit complex non-linear repayment patterns.
- **Benchmarking Role**: This baseline sets the benchmark against which non-linear models (Random Forest, XGBoost) are compared in Step 2.

---
## SECTION 10 — Phase 3 Step 1 Completion Summary

### 📋 Model Development Summary:

| Specification | Value | Context |
|---|---|---|
| **Model Architecture** | `LogisticRegression(max_iter=1000, random_state=42)` | First baseline benchmark |
| **Training Records** | **40** | 80% stratified split |
| **Testing Records** | **10** | 20% stratified split |
| **Input Features** | **16** | 13 scaled numerical + 3 encoded categorical |
| **Target Variable** | `default` | 0 = Non-default, 1 = Default |
| **Outputs Generated** | `y_test_pred` & `y_test_proba` | Discrete class & estimated Probability of Default (PD) |
| **Actual Evaluation Metrics** | Dynamically computed from test predictions | Observed on small 10-record educational test split |

---

```
============================================================
  PHASE 3 — STEP 1 COMPLETE.
  PHASE 3 — STEP 2 NOT STARTED.
============================================================
```